# Pre-processing (occupancy)

**Functions:** the building's occupancy sensors produce one big spreadsheet (a CSV file) mixing
several rooms together. This notebook checks that the file looks exactly as expected, then splits it into one small
file per room, keeping only the two columns we need: the headcount and the time. The room is identified by the file's name.

**Rule:** if a file's column headings are not exactly the expected ones, the code stops with an error instead of guessing.

| Step | Function | What goes in | What comes out |
|---|---|---|---|
| 1 | `check_headings` | a CSV file | nothing, or an error if the headings are wrong |
| 2 | `find_occupancy_file` | the `data` folder | the newest occupancy file in it |
| 3 | `clean_headcount` | a column of headcounts as text | the same counts as numbers |
| 4 | `split_by_room` | the occupancy file | one CSV per room (room ID still included) + a summary |
| 5 | `drop_room_id` | the folder of room files | the same files without the room ID column |
| 6 | `find_floorspace_ids` | room names + the room lookup file | the floorspace ID of each covered room (others are reported) |
| 7 | run | the `data` folder and room names | everything in `derived/preprocessed/`, plus the chosen rooms' IDs |

In [ ]:
# Setup: folders and the exact column headings we expect. Run from the project's main folder.
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd() / "data"
OUTPUT_DIR = Path.cwd() / "derived" / "preprocessed"

# Files are read in pieces of this many rows so the computer's memory is never overloaded.
ROWS_PER_PIECE = 50_000

# The exact headings (and order) of a raw occupancy file.
RAW_HEADINGS = ["id", "deviceid", "floorspaceid", "occupancystatus", "headcount",
                "collecteddate", "occupancystatuschangedate", "previousoccupancystatus"]

# The exact headings of the room lookup file.
LOOKUP_HEADINGS = ["Building", "Floor/Room", "Area", "ID", "Note", "FloorId", "SpaceId"]

# The columns we read from the raw file. The row "id" is never used, so it is not even read.
KEPT_HEADINGS = ["floorspaceid", "headcount", "collecteddate"]

# The final columns of each room's file. The room ID is only needed to split the rooms apart.
FINAL_HEADINGS = ["headcount", "collecteddate"]

In [ ]:
def check_headings(csv_file, expected_headings):
    """Stop with an error if a file's column headings are not exactly as expected.

    Only the heading row is read, so this is fast even for huge files.

    Input:
        csv_file: the file to check.
        expected_headings: the list of headings the file must have, in order.

    Output:
        Nothing if the headings match exactly.

    Errors:
        ValueError if any heading is missing, extra, renamed or in a different order.
    """
    found = list(pd.read_csv(csv_file, nrows=0).columns)
    if found != list(expected_headings):
        raise ValueError(f"{Path(csv_file).name} has the wrong column headings.\n"
                         f"  Expected: {list(expected_headings)}\n  Found:    {found}")

In [ ]:
def find_occupancy_file(data_folder):
    """Find the newest occupancy file inside the data folder (including sub-folders).

    A CSV is treated as occupancy data if it contains a "floorspaceid" heading.
    Other kinds of CSV files in the folder are ignored. If an occupancy file has the
    wrong headings, the code stops with an error rather than using it.

    Input:
        data_folder: the folder that holds the raw data files.

    Output:
        The path of the most recently modified occupancy file.

    Errors:
        FileNotFoundError if no occupancy file is found.
        ValueError if an occupancy file has the wrong headings.
    """
    candidates = []
    for path in sorted(Path(data_folder).rglob("*.csv")):
        if "floorspaceid" in pd.read_csv(path, nrows=0).columns:
            check_headings(path, RAW_HEADINGS)
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f"No occupancy file found in {data_folder}")
    return max(candidates, key=lambda p: p.stat().st_mtime_ns)

In [ ]:
def clean_headcount(headcounts):
    """Turn a column of headcounts (text) into numbers, leaving blanks as "missing".

    A blank (or the words nan / null / none) becomes a missing value, never zero.

    Input:
        headcounts: a column of headcounts as text.

    Output:
        The same column as numbers, with missing values left empty.

    Errors:
        ValueError if any entry is not a whole number of 0 or more.
    """
    text = headcounts.str.strip().str.casefold()
    is_blank = text.isin(["", "nan", "null", "none"])
    numbers = pd.to_numeric(text.mask(is_blank), errors="coerce")
    if (numbers.isna() & ~is_blank).any() or (numbers < 0).any() or ((numbers % 1) != 0).any():
        raise ValueError("Headcounts must be whole numbers of 0 or more (or blank).")
    return numbers

In [ ]:
def split_by_room(occupancy_file, output_folder):
    """Check the occupancy file and split it into one CSV file per room.

    Every row is kept (nothing is deleted). The unused "id" column is dropped straight
    away; the room ID is kept for now and removed later by drop_room_id. Old room
    files already in the output folder are replaced.

    Checks made on every row: the room ID is not blank, the collection date is a
    real date, and the headcount is valid (see clean_headcount).

    Input:
        occupancy_file: the raw occupancy CSV (must have exactly RAW_HEADINGS).
        output_folder: where the room files are saved (created if needed).

    Output:
        A table with one line per room and its number of rows, which is also saved
        as "rows_per_room.csv" in the output folder.

    Errors:
        ValueError if the headings are wrong, a check fails, or the file is empty.
    """
    check_headings(occupancy_file, RAW_HEADINGS)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    for old_file in output_folder.glob("*.csv"):
        old_file.unlink()

    rows_per_room = {}
    for piece in pd.read_csv(occupancy_file, usecols=KEPT_HEADINGS, dtype=str,
                             keep_default_na=False, chunksize=ROWS_PER_PIECE):
        if (piece["floorspaceid"].str.strip() == "").any():
            raise ValueError("Found a row with no room ID.")
        if pd.to_datetime(piece["collecteddate"], errors="coerce", utc=True).isna().any():
            raise ValueError("Found a row with an invalid collection date.")
        piece["headcount"] = clean_headcount(piece["headcount"])
        for room, rows in piece[KEPT_HEADINGS].groupby("floorspaceid"):
            room_file = output_folder / f"{room}.csv"
            rows.to_csv(room_file, mode="a", header=not room_file.exists(), index=False)
            rows_per_room[room] = rows_per_room.get(room, 0) + len(rows)

    if not rows_per_room:
        raise ValueError("The occupancy file contains no data rows.")
    summary = pd.DataFrame({"room_id": list(rows_per_room), "rows": list(rows_per_room.values())})
    summary.to_csv(output_folder / "rows_per_room.csv", index=False)
    return summary

In [ ]:
def drop_room_id(room_folder):
    """Remove the room ID column from every room file, leaving only headcount and time.

    This is the very last step. Each file's name is still the room ID, so no
    information about which room it belongs to is lost. "rows_per_room.csv" is left alone.

    Input:
        room_folder: the folder of room files made by split_by_room.

    Output:
        Nothing. The files are rewritten in place.

    Errors:
        ValueError if a room file does not have the expected headings.
    """
    for room_file in Path(room_folder).glob("*.csv"):
        if room_file.name == "rows_per_room.csv":
            continue
        check_headings(room_file, KEPT_HEADINGS)
        temporary = room_file.with_suffix(".tmp")
        for number, piece in enumerate(pd.read_csv(room_file, usecols=FINAL_HEADINGS, dtype=str,
                                                   keep_default_na=False, chunksize=ROWS_PER_PIECE)):
            piece[FINAL_HEADINGS].to_csv(temporary, mode="a" if number else "w", header=not number, index=False)
        temporary.replace(room_file)

In [ ]:
def find_floorspace_ids(room_names, lookup_folders, streams_folder):
    """Turn room names (e.g. "G.20") into the floorspace IDs used in the occupancy data.

    Each name is matched against the "Floor/Room" column of the room lookup file (the
    file whose name starts with "Sensor ID and Locations - OCC"), using every row of
    the file. Capital letters and spaces around a name do not matter. A room is
    "covered" if it has a floorspace ID in the lookup AND occupancy data exists for it.
    Rooms that are not covered are reported in a message and skipped, so the analysis
    can carry on with the rooms that are covered.

    Input:
        room_names: the rooms the user is interested in, e.g. ["G.20", "G.25"].
        lookup_folders: folders to search for the lookup file (e.g. the data folder
            and the project's main folder).
        streams_folder: the folder of room files made by split_by_room.

    Output:
        A dictionary of {room name: floorspace ID} for the covered rooms only. Each
        room's data file is streams_folder / f"{id}.csv".

    Errors:
        FileNotFoundError if the lookup file is missing.
        ValueError if the lookup headings are wrong, a room name matches more than one
            floorspace ID (ambiguous), or none of the requested rooms are covered.
    """
    lookup_files = [p for folder in lookup_folders for p in Path(folder).glob("Sensor ID and Locations - OCC*.csv")]
    if not lookup_files:
        raise FileNotFoundError("Room lookup file (Sensor ID and Locations - OCC...) not found.")
    lookup_file = max(lookup_files, key=lambda p: p.stat().st_mtime_ns)
    check_headings(lookup_file, LOOKUP_HEADINGS)
    lookup = pd.read_csv(lookup_file, dtype=str, keep_default_na=False)
    lookup_names = lookup["Floor/Room"].str.strip().str.casefold()

    covered = {}
    for room in room_names:
        ids = sorted(set(lookup.loc[lookup_names == room.strip().casefold(), "SpaceId"].str.strip()) - {""})
        if len(ids) > 1:
            raise ValueError(f"Room {room!r} matches more than one floorspace ID: {ids}")
        if not ids:
            print(f"NOT COVERED: {room!r} has no floorspace ID in the lookup file.")
        elif not (Path(streams_folder) / f"{ids[0]}.csv").is_file():
            print(f"NOT COVERED: {room!r} has floorspace ID {ids[0]}, but there is no occupancy data for it.")
        else:
            covered[room] = ids[0]
    if not covered:
        raise ValueError("None of the requested rooms are covered, so there is nothing to analyse.")
    return covered

In [ ]:
# Run the whole preprocessing: the only input is the data folder.
occupancy_file = find_occupancy_file(DATA_DIR)
summary = split_by_room(occupancy_file, OUTPUT_DIR / "occupancy_streams")
drop_room_id(OUTPUT_DIR / "occupancy_streams")
print(f"Used: {occupancy_file.name}")
print(f"Rooms: {len(summary)}   Rows: {summary['rows'].sum():,}")
print(summary.to_string(index=False))

# Choose rooms by name (the "Floor/Room" column of the lookup file). Uncovered rooms are reported and skipped.
ROOM_NAMES = ["G.20", "G.25", "G.40"]
floorspace_ids = find_floorspace_ids(ROOM_NAMES, [DATA_DIR, Path.cwd()], OUTPUT_DIR / "occupancy_streams")
for room, floorspace_id in floorspace_ids.items():
    print(f"Room {room} -> floorspace ID {floorspace_id}")